In [ ]:
import pandas as pd
import pickle
import arviz as az
from plotly.subplots import make_subplots
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.inputs import ANALYSIS_TYPES, get_latest_analyses
from emu_renewal.outputs import get_output_dir, load_targets

set_matplotlib_formats("svg")

In [ ]:
country = "France"
analysis_times = get_latest_analyses("France", ANALYSIS_TYPES)

In [ ]:
spaghs = {}
targets = {}
idatas = {}
mobs = {}
for analysis, time in analysis_times.items():
    outdir = get_output_dir(country, analysis, time)
    spaghs[analysis] = pd.read_hdf(outdir / "spaghetti.h5")
    targets[analysis] = load_targets(country, analysis, time)
    idatas[analysis] = az.from_netcdf(outdir / "idata.nc")

In [ ]:
targ_names = targets["no_mob"].keys()
fig, axes = plt.subplots(len(targ_names), len(analysis_times), figsize=(10, 15), sharex=True, sharey="row")
for o, out in enumerate(targ_names):
    for a, analysis in enumerate(analysis_times):
        ax = axes[o, a]
        for col in spaghs[analysis][out].columns:
            ax.plot(spaghs[analysis].index, spaghs[analysis][out, col], color="black", linewidth=0.2)
        target = targets[analysis][out]
        ax.plot(target.index, target, linewidth=0.0, marker=".")
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=70)
        if o == 0:
            ax.set_title(analysis, fontsize=10)
        if a == 0:
            ax.set_ylabel(out, fontsize=10)
plt.subplots_adjust(wspace=0.05)